# Imports

In [7]:
# general imports
from scipy.signal import butter, filtfilt
import os
import mne
import pandas as pd
import numpy as np
import sys
import time

# import from custom script
import basic_mne_functions as bmf
import shared_processing_functions as spf

# import from different directory
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from hdf5_files import Artefacts_Detection as ad

# Functions

In [8]:
def get_egi_file_paths(dir_path):
    file_list = []

    for dir in os.listdir(dir_path):
        if os.path.isdir(os.path.join(dir_path, dir)):
            for file in os.listdir(os.path.join(dir_path, dir)):
                if ".RAW" in file and "MOTOR" in file and not file.startswith("."):
                    file_list.append(os.path.join(dir_path, dir, file))

    return file_list

# Variables

In [9]:
all_channels = ['E26', 'E9', 'E45', 'E81', 'E132', 'E186', 'E101', 'E137']
fpz = 'E26'
cz = ['E9', 'E45', 'E81', 'E132', 'E186']
pz = 'E101'
oz = 'E137'

# Paths

In [10]:
raw_EGI_files = r"D:\MOTORWP4_DATASET\EGI"
output_path = r"D:\dilon_data\MOTORWP4_dataset\sleep-edf_channel_test"

# Main

In [11]:
# get all .RAW filepaths
print("Retrieving list of all egi .RAW files in directory.")
egi_file_list = get_egi_file_paths(raw_EGI_files)
print("Files retrieved.")
print(len(egi_file_list))

Retrieving list of all egi .RAW files in directory.
Files retrieved.
115


In [13]:
# iterating through all files
for egi_file in egi_file_list:
    print(egi_file)
    subject = egi_file.split("\\")[-1].split(" ")[0]
    if os.path.exists(os.path.join(output_path, f"{subject}.edf")):
        continue
    else:
        print(f"Creating .edf file for subject: {subject}")

        start = time.time()
        # load .RAW egi file
        raw_egi = mne.io.read_raw_egi(
            egi_file, 
            preload=False,
            verbose="error"
        )

        end = time.time()
        print(f'Loading the .RAW EGI file took {end-start:.2f} seconds')

        start = time.time()
        # get channel selection
        raw_egi.pick(all_channels)
        raw_egi.load_data()
        end = time.time()
        print(f'Picking channels and loading data took {end-start:.2f} seconds')

        start = time.time()
        # notch filter
        raw_egi.notch_filter(
            freqs=[50], 
            method='spectrum_fit', 
            n_jobs=2, 
            verbose='error'
        )

        # pre-process the data
        raw_egi.filter(
            l_freq=0.1,
            h_freq=100,
            n_jobs=2,
            verbose='error'
        )

        end = time.time()
        print(f'Filtering data took {end-start:.2f} seconds')

        start = time.time()
        # create cz channel
        cz_data = raw_egi.copy().pick(cz).get_data().mean(axis=0, keepdims=True)
        info = mne.create_info(['Cz'], raw_egi.info['sfreq'], 'eeg')
        cz_ch = mne.io.RawArray(cz_data, info)
        # add cz
        raw_egi.add_channels([cz_ch], force_update_info=True)
        # drop ref cz channels
        raw_egi.drop_channels(cz)
        end = time.time()
        print(f'Creating Cz electrode took {end-start:.2f} seconds')

        start = time.time()
        mne.set_bipolar_reference(
            raw_egi,
            anode = [fpz, pz],
            cathode = ['Cz', oz],
            ch_name = ['Fpz-Cz', 'Pz-Oz'],
            copy = False,
            drop_refs = True
        )
        end = time.time()
        print(f'Creating bipolar channels took {end-start:.2f} seconds')

        print(f'Finished processing {subject} - exporting to .edf now\n')

        raw_egi.export(os.path.join(output_path, f"{subject}.edf"), fmt="edf")

D:\MOTORWP4_DATASET\EGI\S40\MOTORWP4_S40_2 20210412 23.RAW
D:\MOTORWP4_DATASET\EGI\S35\MOTORWP4_S35_1 20201211 2224.RAW
D:\MOTORWP4_DATASET\EGI\S35\MOTORWP4_S35_2 20201214 22.RAW
D:\MOTORWP4_DATASET\EGI\S41\MOTORWP4_S41_1 20210416 22.RAW
D:\MOTORWP4_DATASET\EGI\S41\MOTORWP4_S41_2 20210419 22.RAW
D:\MOTORWP4_DATASET\EGI\S90\MOTORWP4_S90_2 20220701 23.RAW
D:\MOTORWP4_DATASET\EGI\S42\MOTORWP4_S42_1 20210423 22.RAW
D:\MOTORWP4_DATASET\EGI\S42\MOTORWP4_S42_2 20210426 22.RAW
D:\MOTORWP4_DATASET\EGI\S36\MOTORWP4_S36_1 20210125 22.RAW
D:\MOTORWP4_DATASET\EGI\S36\MOTORWP4_S36_2 20210128 22.RAW
D:\MOTORWP4_DATASET\EGI\S89\MOTORWP4_S89_2 20220818 22.RAW
D:\MOTORWP4_DATASET\EGI\S43\MOTORWP4_S43_1 20210507 23.RAW
D:\MOTORWP4_DATASET\EGI\S43\MOTORWP4_S43_2 20210510 23.RAW
D:\MOTORWP4_DATASET\EGI\S91\MOTORWP4_S91_1 20220801 22.RAW
D:\MOTORWP4_DATASET\EGI\S91\MOTORWP4_S91_2 20220804 22.RAW
D:\MOTORWP4_DATASET\EGI\S78\MOTORWP4_S78_1 20220629 22.RAW
D:\MOTORWP4_DATASET\EGI\S78\MOTORWP4_S78_2 20220702 23

C:\Users\andri\AppData\Local\Temp\ipykernel_20728\969456491.py:74: RuntimeWarning: EDF format requires equal-length data blocks, so 0.892 seconds of edge values were appended to all channels when writing the final block.
  raw_egi.export(os.path.join(output_path, f"{subject}.edf"), fmt="edf")


D:\MOTORWP4_DATASET\EGI\S61\MOTORWP4_S61_1 20210319 23.RAW
Creating .edf file for subject: MOTORWP4_S61_1
Loading the .RAW EGI file took 146.88 seconds
Reading 0 ... 8453061  =      0.000 ... 33812.244 secs...
Picking channels and loading data took 108.28 seconds
Filtering data took 68.34 seconds
Creating RawArray with float64 data, n_channels=1, n_times=8453062
    Range : 0 ... 8453061 =      0.000 ... 33812.244 secs
Ready.
Creating Cz electrode took 1.19 seconds
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=8453062
    Range : 0 ... 8453061 =      0.000 ... 33812.244 secs
Ready.
Added the following bipolar channels:
Fpz-Cz, Pz-Oz
Creating bipolar channels took 0.51 seconds
Finished processing MOTORWP4_S61_1 - exporting to .edf now



C:\Users\andri\AppData\Local\Temp\ipykernel_20728\969456491.py:74: RuntimeWarning: EDF format requires equal-length data blocks, so 0.752 seconds of edge values were appended to all channels when writing the final block.
  raw_egi.export(os.path.join(output_path, f"{subject}.edf"), fmt="edf")


D:\MOTORWP4_DATASET\EGI\S61\MOTORWP4_S61_2 20210322 22.RAW
Creating .edf file for subject: MOTORWP4_S61_2
Loading the .RAW EGI file took 135.20 seconds
Reading 0 ... 8058828  =      0.000 ... 32235.312 secs...
Picking channels and loading data took 94.71 seconds
Filtering data took 63.75 seconds
Creating RawArray with float64 data, n_channels=1, n_times=8058829
    Range : 0 ... 8058828 =      0.000 ... 32235.312 secs
Ready.
Creating Cz electrode took 0.98 seconds
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=8058829
    Range : 0 ... 8058828 =      0.000 ... 32235.312 secs
Ready.
Added the following bipolar channels:
Fpz-Cz, Pz-Oz
Creating bipolar channels took 0.36 seconds
Finished processing MOTORWP4_S61_2 - exporting to .edf now



...

C:\Users\andri\AppData\Local\Temp\ipykernel_20728\969456491.py:74: RuntimeWarning: EDF format requires equal-length data blocks, so 0.344 seconds of edge values were appended to all channels when writing the final block.
  raw_egi.export(os.path.join(output_path, f"{subject}.edf"), fmt="edf")


D:\MOTORWP4_DATASET\EGI\S52\MOTORWP4_S52_1 20210901 22.RAW
Creating .edf file for subject: MOTORWP4_S52_1
Loading the .RAW EGI file took 86.43 seconds
Reading 0 ... 8201638  =      0.000 ... 32806.552 secs...
Picking channels and loading data took 79.09 seconds
Filtering data took 34.65 seconds
Creating RawArray with float64 data, n_channels=1, n_times=8201639
    Range : 0 ... 8201638 =      0.000 ... 32806.552 secs
Ready.
Creating Cz electrode took 0.67 seconds
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=8201639
    Range : 0 ... 8201638 =      0.000 ... 32806.552 secs
Ready.
Added the following bipolar channels:
Fpz-Cz, Pz-Oz
Creating bipolar channels took 0.35 seconds
Finished processing MOTORWP4_S52_1 - exporting to .edf now



C:\Users\andri\AppData\Local\Temp\ipykernel_20728\969456491.py:74: RuntimeWarning: EDF format requires equal-length data blocks, so 0.444 seconds of edge values were appended to all channels when writing the final block.
  raw_egi.export(os.path.join(output_path, f"{subject}.edf"), fmt="edf")


D:\MOTORWP4_DATASET\EGI\S52\MOTORWP4_S52_2 20210904 22.RAW
Creating .edf file for subject: MOTORWP4_S52_2
Loading the .RAW EGI file took 85.11 seconds
Reading 0 ... 8019183  =      0.000 ... 32076.732 secs...
Picking channels and loading data took 82.05 seconds
Filtering data took 41.95 seconds
Creating RawArray with float64 data, n_channels=1, n_times=8019184
    Range : 0 ... 8019183 =      0.000 ... 32076.732 secs
Ready.
Creating Cz electrode took 0.74 seconds
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=8019184
    Range : 0 ... 8019183 =      0.000 ... 32076.732 secs
Ready.
Added the following bipolar channels:
Fpz-Cz, Pz-Oz
Creating bipolar channels took 0.33 seconds
Finished processing MOTORWP4_S52_2 - exporting to .edf now



C:\Users\andri\AppData\Local\Temp\ipykernel_20728\969456491.py:74: RuntimeWarning: EDF format requires equal-length data blocks, so 0.264 seconds of edge values were appended to all channels when writing the final block.
  raw_egi.export(os.path.join(output_path, f"{subject}.edf"), fmt="edf")
